# Batch Expansion & Azure Classification

This notebook reads all `classified_index` + `stored_results` data from the source DB,
expands every matched snippet to its full surrounding clause **and** classifies it in a
single LLM call (the LLM decides clause boundaries instead of regex), then writes
everything into a new database named `icat-v2-beta`.

Uses DB-backed streaming for checkpoint/resume: a server-side cursor streams source rows,
and a batch existence check against the target DB skips already-processed rows.
No in-memory `processed_ids` set or `work_queue` list.

In [1]:
# Cell 1: Imports & Configuration
import sys
import os
import json
import time
from datetime import datetime

# Ensure imports resolve to this directory's modules (not the project root)
_nb_dir = os.path.dirname(os.path.abspath("__file__"))
if os.path.basename(os.getcwd()) != "data-ingestion":
    _nb_dir = os.path.join(os.getcwd(), "data-ingestion")
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)
os.chdir(_nb_dir)

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

import insert_data
import pipelineoperation

STATS_FILE = "stats_v2_beta.json"
BATCH_SIZE = 10
TARGET_DB_NAME = "icat-v2-beta"
AZURE_INFERENCE_MODEL = os.environ.get("AZURE_INFERENCE_MODEL")

print("Configuration loaded.")
print(f"  Working directory: {os.getcwd()}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  TARGET_DB_NAME = {TARGET_DB_NAME}")
print(f"  STATS_FILE = {STATS_FILE}")
print(f"  MODEL_NAME = {AZURE_INFERENCE_MODEL}")

Configuration loaded.
  Working directory: /home/sankalp-user/Development/ICAT-SLDA/data-ingestion
  BATCH_SIZE = 10
  TARGET_DB_NAME = icat-v2-beta
  STATS_FILE = stats_v2_beta.json
  MODEL_NAME = Llama-3.3-70B-Instruct


In [2]:
# Cell 2: Database Connections

# Source connection (uses env vars as-is)
source_conn = insert_data.create_connection()
assert source_conn is not None, "Failed to connect to source database"
print(f"Connected to source DB: {os.environ.get('DB_NAME')}")

# Create target database if it doesn't exist
admin_uri = (
    f"host={insert_data.DB_HOST} dbname={insert_data.DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
admin_conn = psycopg.connect(admin_uri, autocommit=True)
with admin_conn.cursor() as cur:
    try:
        cur.execute(f'CREATE DATABASE "{TARGET_DB_NAME}"')
        print(f"Created database '{TARGET_DB_NAME}'")
    except psycopg.errors.DuplicateDatabase:
        print(f"Database '{TARGET_DB_NAME}' already exists")
admin_conn.close()

# Connect to target database
target_uri = (
    f"host={insert_data.DB_HOST} dbname={TARGET_DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
target_conn = psycopg.connect(target_uri)
print(f"Connected to target DB: {TARGET_DB_NAME}")

# Initialize schema in target
insert_data.initialize_db(target_conn)
print("Target DB schema initialized.")

Connected to source DB: icatv1
Database 'icat-v2-beta' already exists
Connected to target DB: icat-v2-beta
Database initialized
Target DB schema initialized.


In [ ]:
# Cell 3: Test Run — Randomised Sample Across Queries
# Combined pipeline: expand + classify + extract discussion, all via LLM.
# All document text stays in PostgreSQL — only small context windows enter Python.
# Snippet-level results go to a temp table; display is a SQL aggregation.

import csv
import gc
import random
import importlib
importlib.reload(pipelineoperation)
from pipelineoperation import expand_and_classify_with_azure, extract_discussion_with_azure

# Pick 5 random queries that have non-empty matching data
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT searchquery FROM classified_index
        WHERE LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2
    """)
    all_queries = [r[0] for r in cur.fetchall()]

test_queries = random.sample(all_queries, min(5, len(all_queries)))
print(f"Randomly selected test queries: {test_queries}")

# Sample up to 4 rows per query directly in SQL
test_rows = []
with source_conn.cursor() as cur:
    for q in test_queries:
        cur.execute("""
            SELECT doc_id, title, searchquery, matching_indents, matching_columns
            FROM classified_index WHERE searchquery = %s
            AND (LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2)
            ORDER BY RANDOM() LIMIT 4
        """, (q,))
        test_rows.extend(cur.fetchall())

print(f"Randomly sampled {len(test_rows)} test rows")

# Temp tables for intermediate results (on target_conn so they survive commits)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS test_run_snippets")
    cur.execute("DROP TABLE IF EXISTS test_run_rows")
    cur.execute("""
        CREATE TEMP TABLE test_run_snippets (
            row_idx INT, doc_id TEXT, query TEXT, title TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    cur.execute("""
        CREATE TEMP TABLE test_run_rows (
            row_idx INT, doc_id TEXT, query TEXT, title TEXT,
            discussion TEXT, sentiment TEXT, sentiment_confidence FLOAT,
            is_contract BOOLEAN
        )
    """)
    target_conn.commit()

# Prepare TSV output file
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = f"test_run_{timestamp}.tsv"
FULL_COLUMNS = [
    'Query', 'DocID', 'Title', 'Snippets_MC', 'Snippets_MI',
    'Expanded_MC', 'Expanded_MI', 'Classified_MC', 'Classified_MI',
    'Num_Snippets_MC', 'Num_Snippets_MI', 'Num_Classified_MC', 'Num_Classified_MI',
    'Avg_Classification_Confidence', 'All_Confidences', 'All_Reasonings',
    'Sentiment', 'Sentiment_Confidence', 'Discussion',
    'Is_Contract', 'IndianKanoon_URL',
]
tsv_file = open(csv_path, 'w', newline='')
tsv_writer = csv.DictWriter(tsv_file, fieldnames=FULL_COLUMNS, delimiter='\t')
tsv_writer.writeheader()
tsv_file.flush()

test_errors = []
rows_written = 0

for row_idx, row in enumerate(test_rows):
    doc_id, title, query, mi_raw, mc_raw = row
    mc = insert_data._safe_eval(mc_raw)
    mi = insert_data._safe_eval(mi_raw)

    try:
        # Process MC snippets — doc_text never enters Python; DB fetch is inside the function
        for s in mc:
            try:
                result = expand_and_classify_with_azure(source_conn, doc_id, s)
            except Exception as e:
                print(f"  [Row {row_idx+1}] expand+classify error for MC snippet in doc {doc_id}: {e}")
                result = {'clause_text': s, 'is_contract_clause': False,
                          'classification_confidence': 0.0, 'classification_reasoning': f'Error: {e}'}
            with target_conn.cursor() as cur:
                cur.execute("""INSERT INTO test_run_snippets
                    (row_idx, doc_id, query, title, snippet_type, snippet_text,
                     clause_text, is_contract_clause, confidence, reasoning)
                    VALUES (%s,%s,%s,%s,'mc',%s,%s,%s,%s,%s)""",
                    (row_idx, doc_id, query, title, s,
                     result['clause_text'], result['is_contract_clause'],
                     result['classification_confidence'], result['classification_reasoning']))
            target_conn.commit()
            del result

        # Process MI snippets
        for s in mi:
            try:
                result = expand_and_classify_with_azure(source_conn, doc_id, s)
            except Exception as e:
                print(f"  [Row {row_idx+1}] expand+classify error for MI snippet in doc {doc_id}: {e}")
                result = {'clause_text': s, 'is_contract_clause': False,
                          'classification_confidence': 0.0, 'classification_reasoning': f'Error: {e}'}
            with target_conn.cursor() as cur:
                cur.execute("""INSERT INTO test_run_snippets
                    (row_idx, doc_id, query, title, snippet_type, snippet_text,
                     clause_text, is_contract_clause, confidence, reasoning)
                    VALUES (%s,%s,%s,%s,'mi',%s,%s,%s,%s,%s)""",
                    (row_idx, doc_id, query, title, s,
                     result['clause_text'], result['is_contract_clause'],
                     result['classification_confidence'], result['classification_reasoning']))
            target_conn.commit()
            del result

        # Fetch first classified clause from temp table for discussion extraction
        with target_conn.cursor() as cur:
            cur.execute("""SELECT clause_text FROM test_run_snippets
                WHERE row_idx = %s AND is_contract_clause = true LIMIT 1""", (row_idx,))
            clause_row = cur.fetchone()

        discussion = ''
        sentiment = ''
        sent_conf = 0.0
        is_contract = False
        if clause_row:
            is_contract = True
            try:
                disc_result = extract_discussion_with_azure(source_conn, doc_id, clause_row[0])
                discussion = disc_result['discussion']
                sentiment = disc_result['sentiment']
                sent_conf = disc_result['sentiment_confidence']
                del disc_result
            except Exception as e:
                print(f"  [Row {row_idx+1}] discussion extraction error for doc {doc_id}: {e}")

        # Insert row-level result
        with target_conn.cursor() as cur:
            cur.execute("""INSERT INTO test_run_rows
                (row_idx, doc_id, query, title, discussion, sentiment,
                 sentiment_confidence, is_contract)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s)""",
                (row_idx, doc_id, query, title, discussion, sentiment,
                 sent_conf, is_contract))
        target_conn.commit()

        # Write TSV row by querying aggregates from temp tables
        with target_conn.cursor() as cur:
            cur.execute("""
                SELECT
                    array_agg(snippet_text) FILTER (WHERE snippet_type='mc') AS snippets_mc,
                    array_agg(snippet_text) FILTER (WHERE snippet_type='mi') AS snippets_mi,
                    array_agg(clause_text) FILTER (WHERE snippet_type='mc') AS expanded_mc,
                    array_agg(clause_text) FILTER (WHERE snippet_type='mi') AS expanded_mi,
                    array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause) AS classified_mc,
                    array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause) AS classified_mi,
                    COUNT(*) FILTER (WHERE snippet_type='mc') AS num_mc,
                    COUNT(*) FILTER (WHERE snippet_type='mi') AS num_mi,
                    COUNT(*) FILTER (WHERE snippet_type='mc' AND is_contract_clause) AS num_cls_mc,
                    COUNT(*) FILTER (WHERE snippet_type='mi' AND is_contract_clause) AS num_cls_mi,
                    ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 4) AS avg_conf,
                    array_agg(confidence) AS all_confs,
                    array_agg(reasoning) AS all_reasonings
                FROM test_run_snippets WHERE row_idx = %s
            """, (row_idx,))
            agg = cur.fetchone()

        tsv_writer.writerow({
            'Query': query, 'DocID': doc_id, 'Title': title or '',
            'Snippets_MC': str(agg[0] or []), 'Snippets_MI': str(agg[1] or []),
            'Expanded_MC': str(agg[2] or []), 'Expanded_MI': str(agg[3] or []),
            'Classified_MC': str(agg[4] or []), 'Classified_MI': str(agg[5] or []),
            'Num_Snippets_MC': agg[6], 'Num_Snippets_MI': agg[7],
            'Num_Classified_MC': agg[8], 'Num_Classified_MI': agg[9],
            'Avg_Classification_Confidence': float(agg[10] or 0),
            'All_Confidences': str(agg[11] or []),
            'All_Reasonings': str(agg[12] or []),
            'Sentiment': sentiment, 'Sentiment_Confidence': round(sent_conf, 4),
            'Discussion': discussion,
            'Is_Contract': 'Yes' if is_contract else 'No',
            'IndianKanoon_URL': f'https://indiankanoon.org/doc/{doc_id}/',
        })
        tsv_file.flush()
        rows_written += 1

        print(f"  Row {row_idx+1}/{len(test_rows)}: doc={doc_id} mc={agg[6]} mi={agg[7]} "
              f"cls_mc={agg[8]} cls_mi={agg[9]} contract={'Yes' if is_contract else 'No'}")

    except Exception as e:
        print(f"  [Row {row_idx+1}] SKIPPED doc {doc_id} (query={query}): {e}")
        test_errors.append({'doc_id': doc_id, 'query': query, 'error': str(e)})

    gc.collect()

tsv_file.close()

# Display summary table via SQL aggregation — no Python-side accumulation
df_display = pd.read_sql("""
    SELECT r.query AS "Query", r.doc_id AS "DocID", LEFT(r.title, 50) AS "Title",
           COUNT(*) FILTER (WHERE s.snippet_type='mc') AS "MC",
           COUNT(*) FILTER (WHERE s.snippet_type='mi') AS "MI",
           COUNT(*) FILTER (WHERE s.snippet_type='mc' AND s.is_contract_clause) AS "Cls MC",
           COUNT(*) FILTER (WHERE s.snippet_type='mi' AND s.is_contract_clause) AS "Cls MI",
           ROUND(AVG(s.confidence) FILTER (WHERE s.is_contract_clause)::numeric, 2) AS "Cls Conf",
           r.sentiment AS "Sentiment",
           ROUND(r.sentiment_confidence::numeric, 2) AS "Sent Conf",
           LEFT(r.discussion, 200) AS "Discussion",
           CASE WHEN r.is_contract THEN 'Yes' ELSE 'No' END AS "Contract?"
    FROM test_run_rows r
    LEFT JOIN test_run_snippets s USING (row_idx)
    GROUP BY r.row_idx, r.query, r.doc_id, r.title,
             r.sentiment, r.sentiment_confidence, r.discussion, r.is_contract
    ORDER BY r.row_idx
""", target_conn)
display(df_display)

if test_errors:
    print(f"\n** {len(test_errors)} row(s) failed: **")
    for err in test_errors:
        print(f"  DocID={err['doc_id']}  query={err['query']}  error={err['error']}")

print(f"\nFull details exported to: {csv_path}")
print(f"  {rows_written} rows ({len(test_errors)} errors), {len(FULL_COLUMNS)} columns")
print("\n** Review the table above. Confidence scores should now appear. **")
print("** Do not proceed to full processing until satisfied. **")

Randomly selected test queries: ['Confidentiality', 'Website terms and conditions', 'Privacy', 'copyright licensing', 'termination']
Randomly sampled 20 test rows
  Row 1/20: doc=1282775 mc=17 mi=0 cls_mc=11 cls_mi=0 contract=Yes
  Row 2/20: doc=1375804 mc=6 mi=2 cls_mc=6 cls_mi=2 contract=Yes


In [ ]:
# Cell 4: Copy Base Tables to Target

# Clear any aborted transaction state, reload module, ensure tables exist
target_conn.rollback()
import importlib
importlib.reload(insert_data)
insert_data.initialize_db(target_conn)

# Copy stored_results using a server-side cursor to avoid loading all rows into RAM
inserted_sr = 0
total_sr = 0
with source_conn.cursor(name="copy_stored_results") as src_cur:
    src_cur.itersize = 100
    src_cur.execute("SELECT Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size FROM stored_results")
    batch = []
    for row in src_cur:
        total_sr += 1
        batch.append(row)
        if len(batch) >= 100:
            with target_conn.cursor() as tgt_cur:
                for r in batch:
                    tgt_cur.execute("""
                        INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
                        VALUES (%s, %s, %s, %s, %s)
                        ON CONFLICT (Doc_ID) DO NOTHING
                    """, r)
                    inserted_sr += tgt_cur.rowcount
            target_conn.commit()
            batch = []
    # Final partial batch
    if batch:
        with target_conn.cursor() as tgt_cur:
            for r in batch:
                tgt_cur.execute("""
                    INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_ID) DO NOTHING
                """, r)
                inserted_sr += tgt_cur.rowcount
        target_conn.commit()

print(f"stored_results: {total_sr} source rows, {inserted_sr} newly inserted into target")

# Copy search_queries
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT searchquery, dateandtime FROM search_queries")
    sq_rows = src_cur.fetchall()

with target_conn.cursor() as tgt_cur:
    for row in sq_rows:
        tgt_cur.execute("""
            INSERT INTO search_queries (searchquery, dateandtime)
            VALUES (%s, %s)
        """, row)
    target_conn.commit()

print(f"search_queries: {len(sq_rows)} rows copied to target")

In [ ]:
# Cell 5: Extract Court Metadata for stored_results
# Populates court_name, judgment_date, case_citation using Indian Kanoon API + LLM fallback.
# Has its own checkpoint file so it can be re-run independently.

import importlib
importlib.reload(pipelineoperation)
importlib.reload(insert_data)

from app import fetch_docmeta
from pipelineoperation import extract_metadata_with_azure

METADATA_CHECKPOINT = "checkpoint_metadata.json"
api_key = os.environ.get("API_KEY")
meta_headers = {'authorization': f"Token {api_key}"}

# Load metadata checkpoint
if os.path.exists(METADATA_CHECKPOINT):
    with open(METADATA_CHECKPOINT, 'r') as f:
        meta_ckpt = json.load(f)
    meta_processed = set(meta_ckpt.get('processed_doc_ids', []))
    print(f"Metadata checkpoint loaded: {len(meta_processed)} docs already processed")
else:
    meta_processed = set()
    print("No metadata checkpoint found. Starting fresh.")

# Find docs needing metadata
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT Doc_ID, Doc_Text FROM stored_results
        WHERE court_name IS NULL OR court_name = ''
    """)
    docs_needing_meta = cur.fetchall()

# Filter out already-checkpointed docs
docs_to_process = [(d, t) for d, t in docs_needing_meta if d not in meta_processed]
print(f"{len(docs_to_process)} docs need metadata ({len(docs_needing_meta)} total without, {len(meta_processed)} already done)")

meta_stats = {'api_success': 0, 'llm_fallback': 0, 'failed': 0}

for i, (doc_id, doc_text) in enumerate(docs_to_process):
    try:
        # Try API first
        meta = fetch_docmeta(doc_id, meta_headers)

        # If court_name still empty, use LLM fallback
        if not meta.get('court_name') and doc_text:
            meta = extract_metadata_with_azure(doc_text[:2000])
            meta_stats['llm_fallback'] += 1
        else:
            meta_stats['api_success'] += 1

        # Update stored_results
        with target_conn.cursor() as cur:
            cur.execute("""
                UPDATE stored_results
                SET court_name = %s, judgment_date = %s, case_citation = %s
                WHERE Doc_ID = %s
            """, (meta.get('court_name', ''), meta.get('judgment_date', ''),
                  meta.get('case_citation', ''), doc_id))

        meta_processed.add(doc_id)

        # Checkpoint every BATCH_SIZE docs
        if (i + 1) % BATCH_SIZE == 0:
            target_conn.commit()
            with open(METADATA_CHECKPOINT, 'w') as f:
                json.dump({'processed_doc_ids': list(meta_processed),
                           'last_updated': datetime.now().isoformat(),
                           'stats': meta_stats}, f, indent=2)
            print(f"  Metadata: {i + 1}/{len(docs_to_process)} processed "
                  f"(API: {meta_stats['api_success']}, LLM: {meta_stats['llm_fallback']})")

        time.sleep(0.2)  # rate limit

    except Exception as e:
        print(f"  Metadata error for {doc_id}: {e}")
        meta_stats['failed'] += 1
        meta_processed.add(doc_id)

# Final commit and checkpoint
target_conn.commit()
with open(METADATA_CHECKPOINT, 'w') as f:
    json.dump({'processed_doc_ids': list(meta_processed),
               'last_updated': datetime.now().isoformat(),
               'stats': meta_stats}, f, indent=2)

print(f"\nMetadata extraction complete:")
print(f"  API success: {meta_stats['api_success']}")
print(f"  LLM fallback: {meta_stats['llm_fallback']}")
print(f"  Failed: {meta_stats['failed']}")

In [ ]:
# Cell 6: Load Stats

if os.path.exists(STATS_FILE):
    with open(STATS_FILE, 'r') as f:
        stats = json.load(f)
    print(f"Stats loaded: {stats}")
else:
    stats = {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_discussions_extracted': 0,
        'total_errors': 0,
    }
    print("No stats file found. Starting fresh.")

In [ ]:
# Cell 7: Streaming Generator — replaces in-memory work_queue

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in the target classified_index.

    Uses a VALUES join so the check is a single round-trip per batch.
    Returns only unprocessed rows as dicts.
    """
    if not rows:
        return []

    # Build pairs for the VALUES clause
    pairs = [(r[0], r[2]) for r in rows]  # (doc_id, searchquery)

    # Build the VALUES placeholders: (%s, %s), (%s, %s), ...
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))
    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]

    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set()
        for row in cur:
            existing.add((row[0], row[1]))

    # Return unprocessed rows as dicts
    result = []
    for r in rows:
        doc_id, title, searchquery, mi_raw, mc_raw, mc_after_raw, mi_after_raw = r
        if (doc_id, searchquery) not in existing:
            result.append({
                'doc_id': doc_id,
                'title': title,
                'searchquery': searchquery,
                'matching_indents': insert_data._safe_eval(mi_raw),
                'matching_columns': insert_data._safe_eval(mc_raw),
                'matching_columns_after_classification': insert_data._safe_eval(mc_after_raw),
                'matching_indents_after_classification': insert_data._safe_eval(mi_after_raw),
            })
    return result


def iter_unprocessed_batches(source_conn, target_conn, batch_size):
    """Generator that streams rows from source classified_index via a server-side cursor
    and yields batches of unprocessed row dicts (filtered against target DB).
    """
    with source_conn.cursor(name="stream_classified_index") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)

        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                unprocessed = _filter_already_processed(target_conn, batch)
                batch = []
                if unprocessed:
                    yield unprocessed

        # Final partial batch
        if batch:
            unprocessed = _filter_already_processed(target_conn, batch)
            if unprocessed:
                yield unprocessed


# Display source and target counts (no full fetch)
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"Source classified_index rows: {source_count}")
print(f"Target classified_index rows: {target_count} (already processed)")
print(f"Approximate remaining: {remaining}")

In [ ]:
# Cell 8: Process — Expand, Classify & Extract Discussion (main loop)
# Per snippet: one LLM call for expand + classify (with confidence scores).
# Per row (if contract clauses found): one LLM call for discussion extraction.
# Run Cell 6 (stats) and Cell 7 (streaming setup) before this cell.
# Doc_Text never enters Python — all text ops are parameterised SQL inside the functions.

import importlib
importlib.reload(pipelineoperation)
from pipelineoperation import expand_and_classify_with_azure, extract_discussion_with_azure

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify_with_azure(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


insert_sql = """
    INSERT INTO classified_index(
        Doc_Id, Title, searchquery,
        matching_indents, matching_columns,
        matching_columns_after_classification,
        matching_indents_after_classification,
        expanded_columns, expanded_indents,
        expanded_columns_after_classification,
        expanded_indents_after_classification,
        extracted_discussion, sentiment,
        classification_confidence, classification_reasoning,
        sentiment_confidence
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
        Title = EXCLUDED.Title,
        matching_indents = EXCLUDED.matching_indents,
        matching_columns = EXCLUDED.matching_columns,
        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
        expanded_columns = EXCLUDED.expanded_columns,
        expanded_indents = EXCLUDED.expanded_indents,
        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
        extracted_discussion = EXCLUDED.extracted_discussion,
        sentiment = EXCLUDED.sentiment,
        classification_confidence = EXCLUDED.classification_confidence,
        classification_reasoning = EXCLUDED.classification_reasoning,
        sentiment_confidence = EXCLUDED.sentiment_confidence
"""

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE})")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE):
    batch_num += 1

    for row in batch:
        try:
            # Process matching_columns snippets — doc_text stays in PostgreSQL
            expanded_columns = []
            expanded_columns_after = []
            mc_after = []
            mc_confidences = []
            mc_reasonings = []
            for s in row['matching_columns']:
                r = expand_classify_with_retry(source_conn, row['doc_id'], s)
                expanded_columns.append(r['clause_text'])
                mc_confidences.append(r['classification_confidence'])
                mc_reasonings.append(r['classification_reasoning'])
                if r['is_contract_clause']:
                    expanded_columns_after.append(r['clause_text'])
                    mc_after.append(s)
                del r
                time.sleep(0.1)

            # Process matching_indents snippets
            expanded_indents = []
            expanded_indents_after = []
            mi_after = []
            mi_confidences = []
            mi_reasonings = []
            for s in row['matching_indents']:
                r = expand_classify_with_retry(source_conn, row['doc_id'], s)
                expanded_indents.append(r['clause_text'])
                mi_confidences.append(r['classification_confidence'])
                mi_reasonings.append(r['classification_reasoning'])
                if r['is_contract_clause']:
                    expanded_indents_after.append(r['clause_text'])
                    mi_after.append(s)
                del r
                time.sleep(0.1)

            # Compute average classification confidence
            all_confidences = mc_confidences + mi_confidences
            all_reasonings = mc_reasonings + mi_reasonings
            classified_confidences = [c for c, a in zip(
                mc_confidences + mi_confidences,
                [s in mc_after for s in row['matching_columns']] +
                [s in mi_after for s in row['matching_indents']]
            ) if a]

            if classified_confidences:
                avg_confidence = sum(classified_confidences) / len(classified_confidences)
                first_reasoning = next(
                    (r for c, r, a in zip(all_confidences, all_reasonings,
                        [s in mc_after for s in row['matching_columns']] +
                        [s in mi_after for s in row['matching_indents']]) if a),
                    all_reasonings[0] if all_reasonings else ''
                )
            elif all_confidences:
                avg_confidence = sum(all_confidences) / len(all_confidences)
                first_reasoning = all_reasonings[0] if all_reasonings else ''
            else:
                avg_confidence = 0.0
                first_reasoning = ''

            del all_confidences, classified_confidences, mc_confidences, mi_confidences
            del all_reasonings, mc_reasonings, mi_reasonings

            # Extract court discussion if any clauses classified as contract
            all_classified = expanded_columns_after + expanded_indents_after
            if all_classified:
                disc_result = extract_discussion_with_retry(
                    source_conn, row['doc_id'], all_classified[0])
                extracted_discussion = disc_result['discussion']
                sentiment = disc_result['sentiment']
                sentiment_conf = disc_result['sentiment_confidence']
                stats['total_discussions_extracted'] += 1
                del disc_result
                time.sleep(0.1)
            else:
                extracted_discussion = ''
                sentiment = ''
                sentiment_conf = 0.0

            # Insert into target DB (16 columns)
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute(insert_sql, (
                    row['doc_id'],
                    row['title'],
                    row['searchquery'],
                    str(row['matching_indents']),
                    str(row['matching_columns']),
                    str(mc_after),
                    str(mi_after),
                    str(expanded_columns),
                    str(expanded_indents),
                    str(expanded_columns_after),
                    str(expanded_indents_after),
                    extracted_discussion,
                    sentiment,
                    str(round(avg_confidence, 3)),
                    first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # Track stats
            stats['total_expanded'] += len(expanded_columns) + len(expanded_indents)
            stats['total_classified_as_clause'] += (
                len(expanded_columns_after) + len(expanded_indents_after)
            )
            stats['total_processed'] += 1

            # Free remaining per-row data
            del expanded_columns, expanded_indents, expanded_columns_after, expanded_indents_after
            del mc_after, mi_after, extracted_discussion

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1

    # Commit and save stats after each batch
    target_conn.commit()
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")

In [ ]:
# Cell 9: Verification

# Row counts in target
with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    target_sr_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM search_queries")
    target_sq_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    target_disc_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_confidence IS NOT NULL AND classification_confidence != ''")
    target_conf_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results WHERE court_name IS NOT NULL AND court_name != ''")
    target_meta_count = cur.fetchone()[0]

# Row counts in source
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    source_sr_count = cur.fetchone()[0]

print("=== Row Count Comparison ===")
print(f"  classified_index:  source={source_ci_count}  target={target_ci_count}")
print(f"  stored_results:    source={source_sr_count}  target={target_sr_count}")
print(f"  search_queries:    target={target_sq_count}")
print(f"  with discussion:   target={target_disc_count}")
print(f"  with confidence:   target={target_conf_count}")
print(f"  with court meta:   target={target_meta_count}")

# Sentiment distribution with confidence
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT sentiment, COUNT(*),
               AVG(CASE WHEN sentiment_confidence != '' THEN CAST(sentiment_confidence AS FLOAT) ELSE NULL END)
        FROM classified_index
        WHERE LENGTH(extracted_discussion) > 0
        GROUP BY sentiment
    """)
    sentiment_dist = cur.fetchall()

if sentiment_dist:
    print("\n=== Sentiment Distribution (with avg confidence) ===")
    for sentiment, count, avg_conf in sentiment_dist:
        avg_str = f"{avg_conf:.2f}" if avg_conf else "N/A"
        print(f"  {sentiment or '(empty)'}: {count} (avg confidence: {avg_str})")

# Classification confidence distribution
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT 
            CASE 
                WHEN CAST(classification_confidence AS FLOAT) >= 0.8 THEN 'high (>=0.8)'
                WHEN CAST(classification_confidence AS FLOAT) >= 0.5 THEN 'medium (0.5-0.8)'
                ELSE 'low (<0.5)'
            END as conf_bucket,
            COUNT(*)
        FROM classified_index
        WHERE classification_confidence IS NOT NULL AND classification_confidence != ''
        GROUP BY conf_bucket
    """)
    conf_dist = cur.fetchall()

if conf_dist:
    print("\n=== Classification Confidence Distribution ===")
    for bucket, count in conf_dist:
        print(f"  {bucket}: {count}")

# Show sample rows with all new fields
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT ci.doc_id, ci.searchquery, ci.sentiment,
               ci.classification_confidence, ci.classification_reasoning,
               ci.sentiment_confidence,
               sr.court_name, sr.judgment_date, sr.case_citation
        FROM classified_index ci
        LEFT JOIN stored_results sr ON ci.doc_id = sr.Doc_ID
        WHERE LENGTH(ci.extracted_discussion) > 0
        LIMIT 10
    """)
    verification_rows = cur.fetchall()

if verification_rows:
    print("\n=== Sample Rows with All New Fields ===")
    for r in verification_rows:
        print(f"  DocID: {r[0]}, Query: {r[1]}")
        print(f"    Sentiment: {r[2]}, Confidence: {r[5]}")
        print(f"    Classification Confidence: {r[3]}, Reasoning: {(r[4] or '')[:80]}")
        print(f"    Court: {r[6]}, Date: {r[7]}, Citation: {(r[8] or '')[:60]}")
        print()
else:
    print("\nNo rows with extracted discussions found.")

# Close connections
source_conn.close()
target_conn.close()
print("\nBoth database connections closed.")